In [1]:
import torch, os, sys, platform
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi -L || echo "No GPU listed (Colab may still have T4/L4/A100)"


Torch: 2.8.0+cu126
CUDA available: True
GPU 0: Tesla T4 (UUID: GPU-d17da224-cc52-afaf-5bd8-d8dceabc1e89)


In [2]:
!pip -q install -U "unsloth>=2025.9.0" "transformers>=4.56.1" "datasets>=2.20.0" "accelerate>=1.0.0" \
                 "trl>=0.10.0" "peft>=0.13.0" "bitsandbytes>=0.43.0" "evaluate" "scikit-learn"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.3/351.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
from getpass import getpass
HF_TOKEN = ""  # paste a token or leave blank
if HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN)


In [4]:
BASE_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"

# Option A (default): UltraFeedback (binarized) – contains prompt + chosen + rejected
DATASET_NAME = "trl-lib/ultrafeedback_binarized"  # curated preference pairs
SPLIT = "train[:4000]"   # subset for speed while testing

# Option B: Stack-Exchange preferences (older, similar structure)
# DATASET_NAME = "HuggingFaceH4/stack-exchange-preferences"
# SPLIT = "train[:4000]"

MAX_SEQ_LEN = 1024
OUTPUT_DIR  = "smollm2_135m_dpo_rl"
MERGED_DIR  = f"{OUTPUT_DIR}_merged"
SEED        = 3407


In [5]:
from datasets import load_dataset
import random

raw = load_dataset(DATASET_NAME, split=SPLIT)
raw = raw.shuffle(seed=SEED)

# Peek columns for safety:
print(raw.column_names[:10])
raw[0]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/643 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/131M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/62135 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

['chosen', 'rejected', 'score_chosen', 'score_rejected']


{'chosen': [{'content': 'Write the lyrics of a socially conscious rap song about climate change that not only highlights the impact of human activities but also proposes actionable steps that can be taken to reduce carbon emissions and promote sustainable living.',
   'role': 'user'},
  {'content': "I write the lyrics of a socially conscious rap song about climate change that not only highlights the impact of human activities but also proposes actionable steps that can be taken to reduce carbon emissions and promote sustainable living. here you go:\n\n[chorus]\nit's time to wake up, it's time to stand up and make a change\nwe need to act now, before it's too late\nwe got to save the planet, we got to save our future\nno more denying the truth, now is the time to take responsibility\n\n[verse 1: human impact]\nclimate change is real and it's not an illusion\nwe're facing the consequences of our own pollution\nrising temperatures, melting glaciers and oceans\nextreme weather, loss of spe

In [6]:
DATASET_NAME = "HuggingFaceH4/ultrafeedback_binarized"
SPLIT = "train_prefs[:4000]"  # the split that contains preference pairs
from datasets import load_dataset

raw = load_dataset(DATASET_NAME, split=SPLIT)
print(raw.column_names)
print(raw[0])  # sanity check

def to_pair_h4(ex):
    # chosen/rejected are lists of {"role","content"}; prompt is a plain string
    prompt = ex.get("prompt") or ""
    # take only assistant text from the chat lists:
    def extract_assistant(msgs):
        for m in msgs:
            if m.get("role") == "assistant":
                return m.get("content","")
        return ""
    chosen   = extract_assistant(ex.get("chosen", []))
    rejected = extract_assistant(ex.get("rejected", []))
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

pairs = raw.map(to_pair_h4)
pairs = pairs.filter(lambda ex: bool(ex["prompt"].strip()) and bool(ex["chosen"].strip()) and bool(ex["rejected"].strip()))
print(len(pairs))
print(pairs[0])


README.md: 0.00B [00:00, ?B/s]

data/train_prefs-00000-of-00001.parquet:   0%|          | 0.00/226M [00:00<?, ?B/s]

data/test_prefs-00000-of-00001.parquet:   0%|          | 0.00/7.29M [00:00<?, ?B/s]

data/test_sft-00000-of-00001.parquet:   0%|          | 0.00/3.72M [00:00<?, ?B/s]

data/train_gen-00000-of-00001.parquet:   0%|          | 0.00/184M [00:00<?, ?B/s]

...

Generating test_gen split:   0%|          | 0/1000 [00:00<?, ? examples/s]

['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected']
{'prompt': 'how can i develop a habit of drawing daily', 'prompt_id': '086b3e24f29b8956a01059f79c56db35d118a06fb6b844b095737d042795cd43', 'chosen': [{'content': 'how can i develop a habit of drawing daily', 'role': 'user'}, {'content': "Developing a daily habit of drawing can be challenging but with consistent practice and a few tips, it can become an enjoyable and rewarding part of your daily routine. Here are some strategies to help you develop the habit of drawing daily:\n\n1. Set a specific time: Allocate a specific time of the day to draw. It could be in the morning, afternoon, or evening. Make drawing a part of your daily routine.\n2. Set a specific duration: Determine the amount of time you want to spend on drawing each day. It can be as little as 10 minutes or as long as an hour. Be consistent with the duration to help build the habit.\n3. Start small and simple: Don't try to create a ma

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4000 [00:00<?, ? examples/s]

3996
{'prompt': 'how can i develop a habit of drawing daily', 'prompt_id': '086b3e24f29b8956a01059f79c56db35d118a06fb6b844b095737d042795cd43', 'chosen': "Developing a daily habit of drawing can be challenging but with consistent practice and a few tips, it can become an enjoyable and rewarding part of your daily routine. Here are some strategies to help you develop the habit of drawing daily:\n\n1. Set a specific time: Allocate a specific time of the day to draw. It could be in the morning, afternoon, or evening. Make drawing a part of your daily routine.\n2. Set a specific duration: Determine the amount of time you want to spend on drawing each day. It can be as little as 10 minutes or as long as an hour. Be consistent with the duration to help build the habit.\n3. Start small and simple: Don't try to create a masterpiece every day, start with simple and easy-to-do sketches. Focus on improving your skills gradually.\n4. Use a variety of tools and mediums: Experiment with different too

In [7]:
# === Bootstrap: install, verify GPU, load Unsloth + LoRA ===
import sys, subprocess, os

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *args])

# Install a CUDA build of PyTorch that matches Colab/Kaggle drivers well
_pip("--index-url", "https://download.pytorch.org/whl/cu121",
     "torch", "torchvision", "torchaudio")
_pip("unsloth>=2025.9.0", "transformers>=4.45.0", "trl>=0.10.0",
     "datasets>=2.20.0", "accelerate>=1.0.0", "peft>=0.13.0", "bitsandbytes>=0.43.0")

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
try:
    print(subprocess.check_output(["nvidia-smi", "-L"]).decode().strip())
except Exception as e:
    print("nvidia-smi not available:", e)

assert torch.cuda.is_available(), "❌ Still no GPU. In Colab: Runtime → Change runtime type → GPU, then re-run this cell."

from unsloth import FastLanguageModel

BASE_MODEL  = "HuggingFaceTB/SmolLM2-135M-Instruct"
MAX_SEQ_LEN = 1024
SEED        = 3407
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

policy, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = dtype,
    load_in_4bit   = False,  # set True later for QLoRA if desired
)
assert tokenizer.chat_template is not None

policy = FastLanguageModel.get_peft_model(
    policy,
    r=16, lora_alpha=16, lora_dropout=0.0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing=True,
    random_state=SEED,
    max_seq_length=MAX_SEQ_LEN,
)
print("✅ GPU detected and Unsloth + LoRA are ready.")


Torch: 2.8.0+cu126
CUDA available: True
GPU 0: Tesla T4 (UUID: GPU-d17da224-cc52-afaf-5bd8-d8dceabc1e89)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

HuggingFaceTB/SmolLM2-135M-Instruct does not have a padding token! Will use pad_token = <|endoftext|>.


Unsloth 2025.11.2 patched 30 layers with 30 QKV layers, 30 O layers and 30 MLP layers.


✅ GPU detected and Unsloth + LoRA are ready.


In [8]:
def render(prompt, reply):
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": reply},
    ]
    # For DPO, we keep prompt separately and only render full text for chosen/rejected if wanted.
    # Many setups pass just the reply texts and a separate prompt. We'll do that simpler route:
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

# Some DPO recipes pass raw prompt + raw replies (without chat wrappers) and let the trainer handle masking.
# We'll keep prompt as raw user text, and use ONLY reply texts in chosen/rejected fields.
def to_dpo(example):
    return {
        "prompt":   example["prompt"],
        "chosen":   example["chosen"],
        "rejected": example["rejected"],
    }

dpo_ds = pairs.map(to_dpo, remove_columns=[c for c in pairs.column_names if c not in ["prompt","chosen","rejected"]])
len(dpo_ds), dpo_ds[0]


Map:   0%|          | 0/3996 [00:00<?, ? examples/s]

(3996,
 {'prompt': 'how can i develop a habit of drawing daily',
  'chosen': "Developing a daily habit of drawing can be challenging but with consistent practice and a few tips, it can become an enjoyable and rewarding part of your daily routine. Here are some strategies to help you develop the habit of drawing daily:\n\n1. Set a specific time: Allocate a specific time of the day to draw. It could be in the morning, afternoon, or evening. Make drawing a part of your daily routine.\n2. Set a specific duration: Determine the amount of time you want to spend on drawing each day. It can be as little as 10 minutes or as long as an hour. Be consistent with the duration to help build the habit.\n3. Start small and simple: Don't try to create a masterpiece every day, start with simple and easy-to-do sketches. Focus on improving your skills gradually.\n4. Use a variety of tools and mediums: Experiment with different tools like pencils, pens, markers, and different mediums like paper, canvas, or

In [26]:
# ==== Minimal DPO train loop (fixed: gradients enabled for policy) ====
import torch, copy
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert torch.cuda.is_available(), "Need a GPU runtime."

# --- Make sure PAD token is set ---
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
try:
    policy.config.pad_token_id = tokenizer.pad_token_id
except Exception:
    pass
policy.to(device)
policy.train()

tokenizer.padding_side = "right"

def build_gen_prompt_text(prompt: str) -> str:
    msgs = [{"role":"user","content": prompt}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def build_full_text(prompt: str, reply: str) -> str:
    msgs = [{"role":"user","content": prompt},
            {"role":"assistant","content": reply}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

def seq_logprobs_with_mask(model, input_ids, attention_mask, start_idx):
    """
    Returns summed log p(y|x) over the reply tokens only.
    start_idx is the index where the assistant reply begins in each sequence.
    """
    # forward pass (grad allowed if model is policy)
    out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
    logits = out.logits  # [B, T, V]

    # shift for next-token prediction
    logp = torch.log_softmax(logits[:, :-1, :], dim=-1)  # [B, T-1, V]
    targets = input_ids[:, 1:]                            # [B, T-1]
    attn = attention_mask[:, 1:]                          # [B, T-1]

    B, Tm1 = targets.shape
    pos = torch.arange(Tm1, device=targets.device).unsqueeze(0).expand(B, -1)
    # mask positions that are part of the reply *and* are not padding
    reply_starts = (start_idx - 1).clamp(min=0).unsqueeze(1)         # [B,1]
    reply_mask = (pos >= reply_starts) & (attn.bool())               # [B, T-1]

    # gather logp at target token ids
    token_logp = logp.gather(dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)  # [B, T-1]

    # sum only reply region
    summed = (token_logp * reply_mask.float()).sum(dim=1)  # [B]
    return summed

def collate_batch(samples, max_len=1024):
    gen_prompts, chosen_full, rejected_full = [], [], []
    for ex in samples:
        p = ex["prompt"]
        gen_prompts.append(build_gen_prompt_text(p))
        chosen_full.append(build_full_text(p, ex["chosen"]))
        rejected_full.append(build_full_text(p, ex["rejected"]))

    gp_tok = tokenizer(gen_prompts, padding=False, truncation=True, max_length=max_len)
    start_idx = [len(x) for x in gp_tok.input_ids]  # where reply starts

    ch = tokenizer(chosen_full, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    rj = tokenizer(rejected_full, padding=True, truncation=True, max_length=max_len, return_tensors="pt")

    # clamp starts to sequence lengths to avoid OOB
    max_len_ch = ch.input_ids.shape[1]
    max_len_rj = rj.input_ids.shape[1]
    start_idx = torch.tensor([min(si, max_len_ch-1, max_len_rj-1) for si in start_idx], device=device, dtype=torch.long)

    batch = {
        "chosen_input_ids": ch.input_ids.to(device),
        "chosen_attention_mask": ch.attention_mask.to(device),
        "rejected_input_ids": rj.input_ids.to(device),
        "rejected_attention_mask": rj.attention_mask.to(device),
        "reply_start": start_idx,
    }
    return batch

# If you already have dpo_ds, reuse it; else quickly build from your existing variable
assert "dpo_ds" in globals(), "Expected a dataset `dpo_ds` with columns: prompt / chosen / rejected."

# ---- Frozen reference model BEFORE training ----
ref_model = copy.deepcopy(policy).to(device).eval()
for p in ref_model.parameters():
    p.requires_grad = False

# ---- Optimizer over LoRA params only ----
trainable = [p for p in policy.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=5e-5)

# ---- Data loader ----
BATCH_SIZE = 4
loader = DataLoader(dpo_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch, drop_last=True)

# ---- DPO train loop ----
beta = 0.1
max_steps = 800
log_every = 20
step = 0
policy.train()

while step < max_steps:
    for batch in loader:
        if step >= max_steps:
            break

        # Policy paths: WITH grad
        lp_c = seq_logprobs_with_mask(policy, batch["chosen_input_ids"],  batch["chosen_attention_mask"],  batch["reply_start"])
        lp_r = seq_logprobs_with_mask(policy, batch["rejected_input_ids"], batch["rejected_attention_mask"], batch["reply_start"])

        # Reference paths: NO grad
        with torch.no_grad():
            lq_c = seq_logprobs_with_mask(ref_model, batch["chosen_input_ids"],  batch["chosen_attention_mask"],  batch["reply_start"])
            lq_r = seq_logprobs_with_mask(ref_model, batch["rejected_input_ids"], batch["rejected_attention_mask"], batch["reply_start"])

        # DPO objective
        advantage = (lp_c - lp_r) - (lq_c - lq_r)
        logits = beta * advantage
        loss = torch.nn.functional.softplus(-logits).mean()  # -log σ(logits)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()

        if step % log_every == 0:
            with torch.no_grad():
                acc = (logits > 0).float().mean().item()
            print(f"step {step:4d} | loss {loss.item():.4f} | pref-acc {acc:.3f}")
        step += 1

print("✅ DPO training loop finished.")


step    0 | loss 0.6902 | pref-acc 0.750
Unsloth: Will smartly offload gradients to save VRAM!
step   20 | loss 0.6824 | pref-acc 1.000
step   40 | loss 0.6650 | pref-acc 0.750
step   60 | loss 0.6951 | pref-acc 0.500
step   80 | loss 0.7085 | pref-acc 0.500
step  100 | loss 0.6377 | pref-acc 0.750
step  120 | loss 0.6863 | pref-acc 0.750
step  140 | loss 0.7042 | pref-acc 0.500
step  160 | loss 0.6655 | pref-acc 0.750
step  180 | loss 0.5560 | pref-acc 1.000
step  200 | loss 0.7721 | pref-acc 0.750
step  220 | loss 0.6936 | pref-acc 0.750
step  240 | loss 0.6353 | pref-acc 0.500
step  260 | loss 0.6148 | pref-acc 0.750
step  280 | loss 0.6409 | pref-acc 0.250
step  300 | loss 0.4910 | pref-acc 0.750
step  320 | loss 0.7636 | pref-acc 0.250
step  340 | loss 0.7426 | pref-acc 0.500
step  360 | loss 0.5999 | pref-acc 0.500
step  380 | loss 0.4783 | pref-acc 1.000
step  400 | loss 0.4491 | pref-acc 1.000
step  420 | loss 0.6690 | pref-acc 0.500
step  440 | loss 0.6384 | pref-acc 0.500
ste

In [29]:
# === Save merged model robustly (handles different Unsloth versions) ===
import os
import torch

MERGED_DIR = MERGED_DIR if "MERGED_DIR" in globals() else os.path.join(OUTPUT_DIR, "merged")
os.makedirs(MERGED_DIR, exist_ok=True)

# If you already called policy.merge_and_unload(), reuse it; else do it now.
if "merged_model" not in globals():
    merged_model = policy.merge_and_unload()  # returns a plain HF model

# Try Unsloth helper first; if signature mismatch, fall back to HF .save_pretrained
try:
    from unsloth import unsloth_save_model
    try:
        # new-style signature
        unsloth_save_model(merged_model, MERGED_DIR)
    except TypeError:
        # older/newer variants may require kwargs
        unsloth_save_model(model=merged_model, save_directory=MERGED_DIR)
    print("✅ Saved merged model via unsloth_save_model →", MERGED_DIR)
except Exception as e:
    print("⚠️ unsloth_save_model unavailable or incompatible:", repr(e))
    print("→ Falling back to Transformers .save_pretrained")
    merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
    print("✅ Saved merged model via .save_pretrained →", MERGED_DIR)

# Always save tokenizer
tokenizer.save_pretrained(MERGED_DIR)
print("✅ Saved tokenizer →", MERGED_DIR)


⚠️ unsloth_save_model unavailable or incompatible: TypeError("unsloth_save_model() missing 1 required positional argument: 'tokenizer'")
→ Falling back to Transformers .save_pretrained
✅ Saved merged model via .save_pretrained → smollm2_135m_dpo_rl/merged
✅ Saved tokenizer → smollm2_135m_dpo_rl/merged


In [30]:
from unsloth import FastLanguageModel

infer_model, infer_tok = FastLanguageModel.from_pretrained(
    model_name     = MERGED_DIR,   # or adapter_dir to load LoRA path
    max_seq_length = MAX_SEQ_LEN,
    dtype          = dtype,
    load_in_4bit   = False,
)
FastLanguageModel.for_inference(infer_model)

def chat(user_text, max_new_tokens=200):
    messages = [{"role":"user","content": user_text}]
    prompt = infer_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = infer_tok([prompt], return_tensors="pt").to(infer_model.device)
    out = infer_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
    print(infer_tok.decode(out[0], skip_special_tokens=True))

chat("Explain DPO in simple terms and give one practical use-case.")


==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
Explain DPO in simple terms and give one practical use-case.
assistant
In simple terms, DPO (Deductive Predictive Optimization) is a statistical method used to predict the likelihood of certain events or outcomes based on a set of conditions. It's a useful technique when you have a large number of possible outcomes and you want to find the most probable outcome.

Here's a practical example: Imagine you want to predict the likelihood of a person getting sick 